# Viscan AI Engineer Round 2 — Bonus: 3D CBCT Segmentation, Nerve Proximity & 3D Mesh Pipeline

**Model:** Pretrained `DentalSegmentator` (`Dataset112_DentalSegmentator_v100.zip`, `nnUNetv2` `3d_fullres`, `fold_0`) used **pretrained as-is** per the assignment brief.  
**Citation:** Dot G, et al. *DentalSegmentator: robust open source deep learning-based CT and CBCT image segmentation.* Journal of Dentistry (2024). DOI: `10.1016/j.jdent.2024.105130` · Weights DOI: `10.5281/zenodo.10829675` (`CC BY 4.0`).  
**Segmented Structures (5):** `1`: Upper Skull / Maxilla · `2`: Mandible · `3`: Upper Teeth · `4`: Lower Teeth · `5`: Mandibular Canal  
**Clinical Finding Post-Processing:** 3D Euclidean minimum distance (`mm`) between `lower_teeth` (`label=4`) and left/right `mandibular_canal` (`label=5`) via `scipy.spatial.cKDTree` in physical (`mm`) space, plus Marching Cubes `.stl` and `.glb` 3D mesh export.

In [1]:
# Cell 1: Install & Verify Dependencies and Hardware Environment
!pip install -q "nnunetv2>=2.2.1,<2.5.0" SimpleITK nibabel scikit-image trimesh scipy psutil matplotlib

import os
import sys
import time
import json
import hashlib
import zipfile
import platform
import urllib.request
from pathlib import Path

import numpy as np
import psutil
import torch
import SimpleITK as sitk
import nibabel as nib
import scipy
from scipy.spatial import cKDTree
from scipy.ndimage import binary_erosion, label as cc_label
from skimage.measure import marching_cubes
import trimesh
import matplotlib.pyplot as plt

WORK_DIR = Path("/kaggle/working/cbct_bonus") if Path("/kaggle/working").exists() else Path("./cbct_bonus")
WORK_DIR.mkdir(parents=True, exist_ok=True)

hw_info = {
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if torch.cuda.is_available() else 0,
    "cpu_count": os.cpu_count(),
    "ram_total_gb": round(psutil.virtual_memory().total / (1024**3), 2),
}
print("=== Hardware & Environment ===")
print(json.dumps(hw_info, indent=2))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
=== Hardware & Environment ===
{
  "python_version": "3.13.15",
  "torch_version": "2.11.0+cu128",
  "cuda_available": true,
  "gpu_name": "Tesla T4",
  "gpu_vram_gb": 14.56,
  "cpu_count": 4,
  "ram_total_gb": 31.35
}


In [2]:
# Cell 2: Download & Verify Official DentalSegmentator Pretrained Weights from Zenodo (Record 10829675)
ZENODO_URL = "https://zenodo.org/api/records/10829675/files/Dataset112_DentalSegmentator_v100.zip/content"
EXPECTED_MD5 = "b71cd5230168d28a4f71b078265b76be"
EXPECTED_BYTES = 229747861

WEIGHTS_DIR = WORK_DIR / "weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
zip_path = WEIGHTS_DIR / "Dataset112_DentalSegmentator_v100.zip"

if not zip_path.exists() or zip_path.stat().st_size != EXPECTED_BYTES:
    print(f"Downloading DentalSegmentator weights from {ZENODO_URL} ...")
    urllib.request.urlretrieve(ZENODO_URL, zip_path)

actual_bytes = zip_path.stat().st_size
md5_hash = hashlib.md5(zip_path.read_bytes()).hexdigest()
print(f"Downloaded: {zip_path.name} | Size: {actual_bytes:,} bytes | MD5: {md5_hash}")
assert actual_bytes == EXPECTED_BYTES, f"Size mismatch: {actual_bytes} != {EXPECTED_BYTES}"
assert md5_hash == EXPECTED_MD5, f"MD5 mismatch: {md5_hash} != {EXPECTED_MD5}"

extract_dir = WEIGHTS_DIR / "extracted"
if not extract_dir.exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

dataset_json_path = next(extract_dir.rglob("dataset.json"))
MODEL_FOLDER = dataset_json_path.parent
dataset_meta = json.loads(dataset_json_path.read_text())
print("\n=== Extracted Model Folder ===")
print("Model folder:", MODEL_FOLDER)
print("dataset.json:", json.dumps(dataset_meta, indent=2))
folds_present = sorted([p.name for p in MODEL_FOLDER.glob("fold_*")])
print("Folds available:", folds_present)

Downloaded: Dataset112_DentalSegmentator_v100.zip | Size: 229,747,861 bytes | MD5: b71cd5230168d28a4f71b078265b76be

=== Extracted Model Folder ===
Model folder: /kaggle/working/cbct_bonus/weights/extracted/Dataset112_DentalSegmentator_v100/nnUNetTrainer__nnUNetPlans__3d_fullres
dataset.json: {
  "channel_names": {
    "0": "CT"
  },
  "labels": {
    "background": 0,
    "Upper Skull": 1,
    "Mandible": 2,
    "Upper Teeth": 3,
    "Lower Teeth": 4,
    "Mandibular canal": 5
  },
  "numTraining": 470,
  "file_ending": ".nii.gz"
}
Folds available: ['fold_0']


In [3]:
# Cell 3: Download Public Dental CBCT Volumes (3D Slicer DentalSurgery Sample Data) & Official Reference Mask
DATA_DIR = WORK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES = {
    "PostDentalSurgery.gipl.gz": (
        "https://github.com/Slicer/SlicerTestingData/releases/download/SHA256/4cdc3dc35519bb57daeef4e5df89c00849750e778809e94971d3876f95cc7bbd",
        "4cdc3dc35519bb57daeef4e5df89c00849750e778809e94971d3876f95cc7bbd",
    ),
    "PreDentalSurgery.gipl.gz": (
        "https://github.com/Slicer/SlicerTestingData/releases/download/SHA256/7bfa16945629c319a439f414cfb7edddd2a97ba97753e12eede3b56a0eb09968",
        "7bfa16945629c319a439f414cfb7edddd2a97ba97753e12eede3b56a0eb09968",
    ),
}

for fname, (url, expected_sha256) in SAMPLES.items():
    fpath = DATA_DIR / fname
    if not fpath.exists():
        urllib.request.urlretrieve(url, fpath)
    sha = hashlib.sha256(fpath.read_bytes()).hexdigest()
    assert sha == expected_sha256, f"SHA256 mismatch for {fname}"
    print(f"Verified {fname}: SHA256={sha[:16]}...")

# Download official reference segmentation from gaudot/SlicerDentalSegmentator
REF_SEG_URL = "https://raw.githubusercontent.com/gaudot/SlicerDentalSegmentator/main/DentalSegmentator/Testing/Data/PostDentalSurgery_Segmentation.nii.gz"
ref_seg_path = DATA_DIR / "PostDentalSurgery_Reference_Segmentation.nii.gz"
if not ref_seg_path.exists():
    urllib.request.urlretrieve(REF_SEG_URL, ref_seg_path)
print(f"Downloaded reference mask: {ref_seg_path.name} ({ref_seg_path.stat().st_size:,} bytes)")

# Convert PostDentalSurgery & PreDentalSurgery to standard NIfTI (.nii.gz) for nnU-Net input
IN_DIR = WORK_DIR / "nnunet_in"
OUT_DIR = WORK_DIR / "nnunet_out"
IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

post_sitk = sitk.ReadImage(str(DATA_DIR / "PostDentalSurgery.gipl.gz"))
post_nii_path = IN_DIR / "PostDentalSurgery_0000.nii.gz"
sitk.WriteImage(post_sitk, str(post_nii_path))

pre_sitk = sitk.ReadImage(str(DATA_DIR / "PreDentalSurgery.gipl.gz"))
pre_nii_path = IN_DIR / "PreDentalSurgery_0000.nii.gz"
sitk.WriteImage(pre_sitk, str(pre_nii_path))

print(f"PostDentalSurgery dimensions: {post_sitk.GetSize()} | spacing (mm): {post_sitk.GetSpacing()}")
print(f"PreDentalSurgery  dimensions: {pre_sitk.GetSize()} | spacing (mm): {pre_sitk.GetSpacing()}")

Verified PostDentalSurgery.gipl.gz: SHA256=4cdc3dc35519bb57...
Verified PreDentalSurgery.gipl.gz: SHA256=7bfa16945629c319...
Downloaded reference mask: PostDentalSurgery_Reference_Segmentation.nii.gz (512,859 bytes)
PostDentalSurgery dimensions: (360, 360, 330) | spacing (mm): (0.5, 0.5, 0.5)
PreDentalSurgery  dimensions: (360, 360, 330) | spacing (mm): (0.5, 0.5, 0.5)


In [5]:
# Cell 4: Run Pretrained DentalSegmentator Inference on GPU & Measure Latency, VRAM, RAM, and Dice vs Reference
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"  # PyTorch >= 2.6 compatibility for nnU-Net v2 checkpoints
os.environ["nnUNet_raw"] = str(WEIGHTS_DIR)
os.environ["nnUNet_preprocessed"] = str(WEIGHTS_DIR)
os.environ["nnUNet_results"] = str(WEIGHTS_DIR)

from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

device = torch.device("cuda", 0) if torch.cuda.is_available() else torch.device("cpu")
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

proc = psutil.Process(os.getpid())
rss_before_mb = proc.memory_info().rss / (1024**2)

t_init_start = time.perf_counter()
predictor = nnUNetPredictor(
    tile_step_size=0.5,
    use_gaussian=True,
    use_mirroring=False,  # Matches SlicerDentalSegmentator Parameter.disableTta = True
    perform_everything_on_device=True if torch.cuda.is_available() else False,
    device=device,
    verbose=False,
    verbose_preprocessing=False,
    allow_tqdm=True,
)
predictor.initialize_from_trained_model_folder(
    str(MODEL_FOLDER),
    use_folds=(0,),
    checkpoint_name="checkpoint_final.pth",
)
init_ms = (time.perf_counter() - t_init_start) * 1000.0

# Predict on PostDentalSurgery_0000.nii.gz
out_post_seg = OUT_DIR / "PostDentalSurgery.nii.gz"
t_inf_start = time.perf_counter()
predictor.predict_from_files(
    [[str(post_nii_path)]],
    [str(out_post_seg)],
    save_probabilities=False,
    overwrite=True,
    num_processes_preprocessing=1,
    num_processes_segmentation_export=1,
)
if torch.cuda.is_available():
    torch.cuda.synchronize()
gpu_inference_ms = (time.perf_counter() - t_inf_start) * 1000.0
peak_vram_mb = (torch.cuda.max_memory_allocated() / (1024**2)) if torch.cuda.is_available() else 0.0
peak_rss_mb = proc.memory_info().rss / (1024**2)

print(f"\n=== GPU Inference Benchmark (PostDentalSurgery: {post_sitk.GetSize()}) ===")
print(f"Model Initialization Time: {init_ms:.1f} ms")
print(f"End-to-End 3D Segmentation Latency: {gpu_inference_ms:.1f} ms ({gpu_inference_ms/1000.0:.2f} s)")
print(f"Peak GPU VRAM Allocated: {peak_vram_mb:.1f} MB")
print(f"Process RSS Memory: {peak_rss_mb:.1f} MB (Delta: +{peak_rss_mb - rss_before_mb:.1f} MB)")

# Verify Dice similarity against SlicerDentalSegmentator's official reference mask
pred_arr = sitk.GetArrayFromImage(sitk.ReadImage(str(out_post_seg)))
ref_arr = sitk.GetArrayFromImage(sitk.ReadImage(str(ref_seg_path)))

CLASS_MAP = {
    1: "maxilla_upper_skull",
    2: "mandible",
    3: "upper_teeth",
    4: "lower_teeth",
    5: "mandibular_canal",
}

dice_By_class = {}
print("\n=== Verification vs. Official SlicerDentalSegmentator Reference Mask ===")
for cid, cname in CLASS_MAP.items():
    p_mask = (pred_arr == cid)
    r_mask = (ref_arr == cid)
    inter = np.logical_and(p_mask, r_mask).sum()
    denom = p_mask.sum() + r_mask.sum()
    dice = float(2.0 * inter / denom) if denom > 0 else 1.0
    dice_By_class[cname] = round(dice, 4)
    print(f"  Class {cid} ({cname:20s}): Pred voxels={int(p_mask.sum()):8d} | Ref voxels={int(r_mask.sum()):8d} | Dice={dice:.4f}")

/usr/local/lib/python3.13/dist-packages/nnunetv2/inference/predict_from_raw_data.py:84: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  checkpoint = torch.load(join(model_training_output_dir, f'fold_{f}', checkpoint_name),
/usr/local/lib/python3.13/dist-packages/nnunetv2/utilities/plans_handling/plans_handler.py:37: UserWarning: Detected old nnU-Net plans format. Attempting to reconstruct network architecture parameters. If this fails, rerun nnUNetv2_plan_experiment for your dataset. If you use a custom architecture, please downgrade nnU-Net to the version you implemented this or update your implementation + plans.
  warnings.warn("Detected old nnU-Net plans format. Attempting to reconstruct network architecture "


There are 1 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 1 cases that I would like to predict

Predicting PostDentalSurgery.nii.gz:
perform_everything_on_device: True


100%|██████████| 252/252 [00:30<00:00,  8.21it/s]


sending off prediction to background worker for resampling and export
done with PostDentalSurgery.nii.gz

=== GPU Inference Benchmark (PostDentalSurgery: (360, 360, 330)) ===
Model Initialization Time: 2131.9 ms
End-to-End 3D Segmentation Latency: 131407.1 ms (131.41 s)
Peak GPU VRAM Allocated: 4936.0 MB
Process RSS Memory: 2641.8 MB (Delta: +1270.8 MB)

=== Verification vs. Official SlicerDentalSegmentator Reference Mask ===
  Class 1 (maxilla_upper_skull ): Pred voxels= 1477665 | Ref voxels= 1446339 | Dice=0.9768
  Class 2 (mandible            ): Pred voxels=  310574 | Ref voxels=  309911 | Dice=0.9849
  Class 3 (upper_teeth         ): Pred voxels=   75741 | Ref voxels=   75300 | Dice=0.9867
  Class 4 (lower_teeth         ): Pred voxels=   64305 | Ref voxels=   63081 | Dice=0.9745
  Class 5 (mandibular_canal    ): Pred voxels=    6063 | Ref voxels=    6655 | Dice=0.8968


In [6]:
# Cell 5: 3D Post-Processing — Island Filtering & Lower Teeth to Mandibular Canal Euclidean Distance (mm)
t_post_start = time.perf_counter()

seg_img = sitk.ReadImage(str(out_post_seg))
vol_img = sitk.ReadImage(str(post_nii_path))
spacing_xyz = seg_img.GetSpacing()  # (sx, sy, sz) in mm
origin_xyz = seg_img.GetOrigin()
direction = np.array(seg_img.GetDirection()).reshape(3, 3)

# sitk.GetArrayFromImage returns (Z, Y, X); transpose to (X, Y, Z) for intuitive indexing
seg_xyz = sitk.GetArrayFromImage(seg_img).transpose(2, 1, 0).copy()
vol_xyz = sitk.GetArrayFromImage(vol_img).transpose(2, 1, 0).copy()
nx, ny, nz = seg_xyz.shape
voxel_vol_mm3 = float(spacing_xyz[0] * spacing_xyz[1] * spacing_xyz[2])

# Post-processing matching SlicerDentalSegmentator (_minimumIslandSize_mm3 = 60 on segments 1..4)
min_voxels = int(np.ceil(60.0 / voxel_vol_mm3))
for cid in [1, 2, 3, 4]:
    mask = (seg_xyz == cid)
    labeled, num_feats = cc_label(mask)
    if num_feats > 0:
        counts = np.bincount(labeled.ravel())
        small_labels = np.where((counts < min_voxels) & (np.arange(len(counts)) > 0))[0]
        if len(small_labels) > 0:
            seg_xyz[np.isin(labeled, small_labels)] = 0

def voxel_to_world_mm(ijk_array):
    """Converts (N, 3) voxel indices (i, j, k) to physical (x, y, z) in mm."""
    scaled = ijk_array * np.array(spacing_xyz, dtype=np.float64)
    return np.array(origin_xyz, dtype=np.float64) + scaled @ direction.T

# Structure volumetric statistics
structure_stats = {}
for cid, cname in CLASS_MAP.items():
    mask = (seg_xyz == cid)
    v_count = int(mask.sum())
    if v_count > 0:
        coords = np.argwhere(mask)
        bbox_vox = [int(coords[:,0].min()), int(coords[:,1].min()), int(coords[:,2].min()),
                    int(coords[:,0].max()), int(coords[:,1].max()), int(coords[:,2].max())]
    else:
        bbox_vox = [0, 0, 0, 0, 0, 0]
    structure_stats[cname] = {
        "label_id": cid,
        "voxel_count": v_count,
        "volume_cm3": round(v_count * voxel_vol_mm3 / 1000.0, 2),
        "bbox_3d_voxel": bbox_vox,
    }

# Extract surface voxels of Lower Teeth (4) and Mandibular Canal (5)
teeth_mask = (seg_xyz == 4)
canal_mask = (seg_xyz == 5)
teeth_surf = teeth_mask & ~binary_erosion(teeth_mask)
canal_surf = canal_mask & ~binary_erosion(canal_mask)

teeth_vox = np.argwhere(teeth_surf)
canal_vox = np.argwhere(canal_surf)

findings = []
mid_x = nx / 2.0
for hemi_name, hemi_filter in [("right", lambda v: v[:, 0] < mid_x), ("left", lambda v: v[:, 0] >= mid_x)]:
    t_v = teeth_vox[hemi_filter(teeth_vox)]
    c_v = canal_vox[hemi_filter(canal_vox)]
    if len(t_v) == 0 or len(c_v) == 0:
        continue
    # Compute distances in isotropic/physical spacing (mm)
    t_phys = t_v * np.array(spacing_xyz, dtype=np.float64)
    c_phys = c_v * np.array(spacing_xyz, dtype=np.float64)
    tree = cKDTree(c_phys)
    dists, idxs = tree.query(t_phys)
    min_idx = int(np.argmin(dists))
    min_dist_mm = float(dists[min_idx])
    root_ijk = t_v[min_idx]
    canal_ijk = c_v[idxs[min_idx]]
    root_mm = voxel_to_world_mm(root_ijk)
    canal_mm = voxel_to_world_mm(canal_ijk)
    roi_min = np.minimum(root_ijk, canal_ijk) - 10
    roi_max = np.maximum(root_ijk, canal_ijk) + 10
    bbox_3d_voxel = [
        int(max(0, roi_min[0])), int(max(0, roi_min[1])), int(max(0, roi_min[2])),
        int(min(nx - 1, roi_max[0])), int(min(ny - 1, roi_max[1])), int(min(nz - 1, roi_max[2])),
    ]
    findings.append({
        "class": "mandibular_canal_to_lower_teeth_clearance",
        "hemimandible": hemi_name,
        "min_distance_mm": round(min_dist_mm, 2),
        "confidence": 1.0,
        "tooth_fdi": None,
        "bbox_3d_voxel": bbox_3d_voxel,
        "closest_lower_tooth_voxel": [int(x) for x in root_ijk],
        "closest_canal_voxel": [int(x) for x in canal_ijk],
        "closest_lower_tooth_mm": [round(float(x), 2) for x in root_mm],
        "closest_canal_mm": [round(float(x), 2) for x in canal_mm],
        "key_slice_indices": {
            "sagittal_x": int(root_ijk[0]),
            "coronal_y": int(root_ijk[1]),
            "axial_z": int(root_ijk[2]),
        }
    })

postprocess_ms = (time.perf_counter() - t_post_start) * 1000.0
print(f"Post-processing completed in {postprocess_ms:.1f} ms")
print(json.dumps(findings, indent=2))

Post-processing completed in 5816.3 ms
[
  {
    "class": "mandibular_canal_to_lower_teeth_clearance",
    "hemimandible": "right",
    "min_distance_mm": 0.71,
    "confidence": 1.0,
    "tooth_fdi": null,
    "bbox_3d_voxel": [
      108,
      141,
      78,
      129,
      161,
      99
    ],
    "closest_lower_tooth_voxel": [
      119,
      151,
      89
    ],
    "closest_canal_voxel": [
      118,
      151,
      88
    ],
    "closest_lower_tooth_mm": [
      59.5,
      75.5,
      44.5
    ],
    "closest_canal_mm": [
      59.0,
      75.5,
      44.0
    ],
    "key_slice_indices": {
      "sagittal_x": 119,
      "coronal_y": 151,
      "axial_z": 89
    }
  },
  {
    "class": "mandibular_canal_to_lower_teeth_clearance",
    "hemimandible": "left",
    "min_distance_mm": 1.58,
    "confidence": 1.0,
    "tooth_fdi": null,
    "bbox_3d_voxel": [
      225,
      122,
      62,
      246,
      142,
      85
    ],
    "closest_lower_tooth_voxel": [
      235,
      1

In [7]:
# Cell 6: Marching Cubes 3D Mesh Generation (.stl & .glb) + Multi-Planar Orthogonal Visualization
MESH_DIR = WORK_DIR / "meshes"
MESH_DIR.mkdir(parents=True, exist_ok=True)

# SlicerDentalSegmentator official RGBA palette
RGBA_COLORS = {
    1: [227, 221, 144, 110],  # Maxilla & Upper Skull (translucent)
    2: [212, 161, 230, 110],  # Mandible (translucent so canal & roots inside bone are visible)
    3: [220, 149, 101, 255],  # Upper Teeth (opaque)
    4: [235, 223, 180, 255],  # Lower Teeth (opaque)
    5: [216, 101,  79, 255],  # Mandibular Canal (high-contrast red-orange, opaque)
}

t_mesh_start = time.perf_counter()
scene = trimesh.Scene()
stl_files = {}

# Downsample factor 2 for fast, browser-friendly WebGL mesh (< 5 MB GLB)
step_ds = 2
eff_spacing = (spacing_xyz[0] * step_ds, spacing_xyz[1] * step_ds, spacing_xyz[2] * step_ds)

for cid, cname in CLASS_MAP.items():
    sub_mask = (seg_xyz[::step_ds, ::step_ds, ::step_ds] == cid).astype(np.uint8)
    if sub_mask.sum() < 10:
        continue
    verts, faces, normals, _ = marching_cubes(sub_mask, level=0.5, spacing=eff_spacing)
    mesh = trimesh.Trimesh(vertices=verts, faces=faces, vertex_normals=normals, process=False)
    mesh.visual.vertex_colors = np.tile(RGBA_COLORS[cid], (len(verts), 1))
    stl_path = MESH_DIR / f"PostDentalSurgery_{cname}.stl"
    mesh.export(str(stl_path))
    stl_files[cname] = stl_path.name
    scene.add_geometry(mesh, node_name=cname, geom_name=cname)
    print(f"Mesh {cname:20s}: {len(verts):6d} vertices | {len(faces):6d} faces -> {stl_path.name} ({stl_path.stat().st_size/1024:.1f} KB)")

glb_path = MESH_DIR / "PostDentalSurgery_scene.glb"
scene.export(str(glb_path))
mesh_ms = (time.perf_counter() - t_mesh_start) * 1000.0
print(f"\nExported combined 3D GLB scene: {glb_path.name} ({glb_path.stat().st_size/1024:.1f} KB) in {mesh_ms:.1f} ms")

# Generate 4-panel Multi-Planar Reconstruction (Axial, Coronal, Sagittal at closest contact + Pseudo-Panoramic MIP)
primary_finding = min(findings, key=lambda f: f["min_distance_mm"])
sx_idx = primary_finding["key_slice_indices"]["sagittal_x"]
cy_idx = primary_finding["key_slice_indices"]["coronal_y"]
az_idx = primary_finding["key_slice_indices"]["axial_z"]

fig, axes = plt.subplots(2, 2, figsize=(14, 11), facecolor="#0F172A")
cmap_overlay = plt.cm.colors.ListedColormap(["none", "#E3DD90", "#D4A1E6", "#DC9565", "#EBDFB4", "#FF3B30"])

# 1. Axial Slice (Z = az_idx)
ax = axes[0, 0]
ax.imshow(vol_xyz[:, :, az_idx].T, cmap="gray", origin="lower")
ax.imshow(np.ma.masked_where(seg_xyz[:, :, az_idx].T == 0, seg_xyz[:, :, az_idx].T), cmap=cmap_overlay, alpha=0.55, vmin=0, vmax=5, origin="lower")
ax.scatter([primary_finding["closest_lower_tooth_voxel"][0], primary_finding["closest_canal_voxel"][0]],
           [primary_finding["closest_lower_tooth_voxel"][1], primary_finding["closest_canal_voxel"][1]],
           c=["#00FF88", "#FF2D55"], s=60, zorder=5)
ax.set_title(f"Axial Slice (Z={az_idx}) — Closest Nerve Clearance: {primary_finding['min_distance_mm']} mm", color="white", fontsize=11, fontweight="bold")
ax.axis("off")

# 2. Coronal Slice (Y = cy_idx)
ax = axes[0, 1]
ax.imshow(vol_xyz[:, cy_idx, :].T, cmap="gray", origin="lower")
ax.imshow(np.ma.masked_where(seg_xyz[:, cy_idx, :].T == 0, seg_xyz[:, cy_idx, :].T), cmap=cmap_overlay, alpha=0.55, vmin=0, vmax=5, origin="lower")
ax.plot([primary_finding["closest_lower_tooth_voxel"][0], primary_finding["closest_canal_voxel"][0]],
        [primary_finding["closest_lower_tooth_voxel"][2], primary_finding["closest_canal_voxel"][2]],
        color="#00FF88", linewidth=2.2)
ax.set_title(f"Coronal Slice (Y={cy_idx}) — Root-to-Canal Vector ({primary_finding['hemimandible'].title()})", color="white", fontsize=11, fontweight="bold")
ax.axis("off")

# 3. Sagittal Slice (X = sx_idx)
ax = axes[1, 0]
ax.imshow(vol_xyz[sx_idx, :, :].T, cmap="gray", origin="lower")
ax.imshow(np.ma.masked_where(seg_xyz[sx_idx, :, :].T == 0, seg_xyz[sx_idx, :, :].T), cmap=cmap_overlay, alpha=0.55, vmin=0, vmax=5, origin="lower")
ax.plot([primary_finding["closest_lower_tooth_voxel"][1], primary_finding["closest_canal_voxel"][1]],
        [primary_finding["closest_lower_tooth_voxel"][2], primary_finding["closest_canal_voxel"][2]],
        color="#00FF88", linewidth=2.2)
ax.set_title(f"Sagittal Slice (X={sx_idx}) — Lower Molar Root & Mandibular Canal", color="white", fontsize=11, fontweight="bold")
ax.axis("off")

# 4. Coronal Maximum Intensity Projection (MIP) with 5-Structure Overlay
ax = axes[1, 1]
mip_img = np.max(vol_xyz, axis=1).T
mip_seg = np.max(seg_xyz, axis=1).T
ax.imshow(mip_img, cmap="gray", origin="lower")
ax.imshow(np.ma.masked_where(mip_seg == 0, mip_seg), cmap=cmap_overlay, alpha=0.50, vmin=0, vmax=5, origin="lower")
ax.set_title("3D Coronal MIP — 5 Segmented Structures (DentalSegmentator)", color="white", fontsize=11, fontweight="bold")
ax.axis("off")

plt.tight_layout()
ortho_png_path = WORK_DIR / "PostDentalSurgery_cbct_ortho.png"
plt.savefig(ortho_png_path, dpi=160, bbox_inches="tight", facecolor="#0F172A")
plt.show()
print(f"Saved multi-planar visualization to {ortho_png_path}")

Mesh maxilla_upper_skull : 181032 vertices | 366752 faces -> PostDentalSurgery_maxilla_upper_skull.stl (17907.9 KB)
Mesh mandible            :  35515 vertices |  71934 faces -> PostDentalSurgery_mandible.stl (3512.5 KB)
Mesh upper_teeth         :   9610 vertices |  19240 faces -> PostDentalSurgery_upper_teeth.stl (939.5 KB)
Mesh lower_teeth         :   8530 vertices |  17068 faces -> PostDentalSurgery_lower_teeth.stl (833.5 KB)
Mesh mandibular_canal    :   1736 vertices |   3456 faces -> PostDentalSurgery_mandibular_canal.stl (168.8 KB)

Exported combined 3D GLB scene: PostDentalSurgery_scene.glb (12076.0 KB) in 938.6 ms
Saved multi-planar visualization to /kaggle/working/cbct_bonus/PostDentalSurgery_cbct_ortho.png


In [8]:
# Cell 7: Package All Verified CBCT Deliverables & Final JSON Report
cbct_report = {
    "image_id": "cbct_post_dental_surgery",
    "modality": "CBCT",
    "model": {
        "name": "DentalSegmentator_nnUNetv2_3d_fullres",
        "version": "v1.0.0 (Dataset112_DentalSegmentator, Dot et al. 2024)",
        "weights_doi": "10.5281/zenodo.10829675",
        "license": "CC BY 4.0",
        "pretrained_as_is": True,
    },
    "hardware_benchmark": {
        **hw_info,
        "model_init_ms": round(init_ms, 1),
        "gpu_inference_ms": round(gpu_inference_ms, 1),
        "postprocessing_ms": round(postprocess_ms, 1),
        "mesh_generation_ms": round(mesh_ms, 1),
        "peak_gpu_vram_mb": round(peak_vram_mb, 1),
        "peak_cpu_rss_mb": round(peak_rss_mb, 1),
    },
    "reference_verification_dice": dice_By_class,
    "volume_metadata": {
        "dimensions_voxels": [int(nx), int(ny), int(nz)],
        "spacing_mm": [float(s) for s in spacing_xyz],
        "segmented_structures": structure_stats,
    },
    "findings": findings,
    "summary": (
        f"CBCT 5-structure segmentation complete (Maxilla, Mandible, Upper Teeth, Lower Teeth, Mandibular Canal). "
        + "; ".join([
            f"{f['hemimandible'].title()} lower teeth-to-mandibular canal minimum clearance: {f['min_distance_mm']:.2f} mm"
            for f in findings
        ]) + "."
    ),
    "annotated_image_url": "/results/cbct_post_dental_surgery_annotated.png",
    "mesh_glb_url": "/results/cbct_post_dental_surgery_scene.glb",
    "mesh_stl_urls": {k: f"/results/{v}" for k, v in stl_files.items()},
    "disclaimer": "AI-assisted findings for dentist review. Not a diagnosis.",
}

report_json_path = WORK_DIR / "PostDentalSurgery_cbct_report.json"
report_json_path.write_text(json.dumps(cbct_report, indent=2))
print("=== Final CBCT Report JSON ===")
print(json.dumps(cbct_report, indent=2))

# Bundle all artifacts into a single zip for easy download from Kaggle
bundle_zip = Path("/kaggle/working/cbct_deliverables.zip") if Path("/kaggle/working").exists() else Path("./cbct_deliverables.zip")
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(report_json_path, arcname="PostDentalSurgery_cbct_report.json")
    zf.write(ortho_png_path, arcname="PostDentalSurgery_cbct_ortho.png")
    zf.write(out_post_seg, arcname="PostDentalSurgery_Segmentation_pred.nii.gz")
    zf.write(glb_path, arcname="PostDentalSurgery_scene.glb")
    for stl_f in MESH_DIR.glob("*.stl"):
        zf.write(stl_f, arcname=stl_f.name)

print(f"\n=== Packaged all deliverables into {bundle_zip} ({bundle_zip.stat().st_size / 1024:.1f} KB) ===")

=== Final CBCT Report JSON ===
{
  "image_id": "cbct_post_dental_surgery",
  "modality": "CBCT",
  "model": {
    "name": "DentalSegmentator_nnUNetv2_3d_fullres",
    "version": "v1.0.0 (Dataset112_DentalSegmentator, Dot et al. 2024)",
    "weights_doi": "10.5281/zenodo.10829675",
    "license": "CC BY 4.0",
    "pretrained_as_is": true
  },
  "hardware_benchmark": {
    "python_version": "3.13.15",
    "torch_version": "2.11.0+cu128",
    "cuda_available": true,
    "gpu_name": "Tesla T4",
    "gpu_vram_gb": 14.56,
    "cpu_count": 4,
    "ram_total_gb": 31.35,
    "model_init_ms": 2131.9,
    "gpu_inference_ms": 131407.1,
    "postprocessing_ms": 5816.3,
    "mesh_generation_ms": 938.6,
    "peak_gpu_vram_mb": 4936.0,
    "peak_cpu_rss_mb": 2641.8
  },
  "reference_verification_dice": {
    "maxilla_upper_skull": 0.9768,
    "mandible": 0.9849,
    "upper_teeth": 0.9867,
    "lower_teeth": 0.9745,
    "mandibular_canal": 0.8968
  },
  "volume_metadata": {
    "dimensions_voxels": [
 